In [ ]:
from _Setup import * 
import ast
import re

In [11]:
synth = pd.read_csv('../Data/synthetic_text.csv')


In [20]:
import pandas as pd
import re

def map_tokens_to_entities_bio_fixed(row):
    # Preprocess entity lists into sets for fast lookup
    target_lists = {
        'EMAIL': set(row['EMAIL']),
        'USERNAME': set(row['USERNAME']),
        'ID_NUM': set(row['ID_NUM']),
        'PHONE_NUM': set(row['PHONE_NUM']),
        'URL_PERSONAL': set(row['URL_PERSONAL']),
        'STREET_ADDRESS': set(row['STREET_ADDRESS'])
    }
    
    # Tokenize essay: smart regex for URLs, phones, words
    tokens = re.findall(r'\+?\(?\d[\d\-x\(\)]{5,}\d|\bhttps?://[^\s]+|\b[\w\.\'-]+', row['Essay'])
    
    labels = ['O'] * len(tokens)
    
    i = 0
    while i < len(tokens):
        matched = False
        
        for entity_type, entity_set in target_lists.items():
            for entity in entity_set:
                # Tokenize entity same way
                entity_tokens = re.findall(r'\+?\(?\d[\d\-x\(\)]{5,}\d|\bhttps?://[^\s]+|\b[\w\.\'-]+', entity)
                entity_len = len(entity_tokens)
                
                # Try to match entity tokens to tokens[i:...]
                if tokens[i:i+entity_len] == entity_tokens:
                    # Label the first token as B-ENTITY
                    labels[i] = f'B-{entity_type}'
                    # Label the subsequent tokens as I-ENTITY
                    for j in range(1, entity_len):
                        if i + j < len(tokens):
                            labels[i+j] = f'I-{entity_type}'
                    i += entity_len
                    matched = True
                    break  # Stop checking entities once matched
            if matched:
                break
        
        if not matched:
            i += 1
    
    return tokens, labels

# Usage Example:
# df = pd.DataFrame(your data)
synth[['tokens', 'labels']] = synth.apply(lambda row: pd.Series(map_tokens_to_entities_bio_fixed(row)), axis=1)


In [21]:
synth.head()

,Essay,EMAIL,USERNAME,ID_NUM,PHONE_NUM,URL_PERSONAL,STREET_ADDRESS,tokens,labels
0,The digital age has blurred the lines between ...,[lisa08@gmail.com],[carrie26],"[711801320, 522 AYO]",[+1-875-586-8809x1891],[https://twitter.com/amanda96],"[9930 Joy Hollow Suite 517\nSherriport, WI 539...","[The, digital, age, has, blurred, the, lines, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
1,The digital age has irrevocably woven itself i...,[tammy76@yahoo.com],"[nhoward, juancampos]",[Z89-24I],[515-978-1565],"[https://twitter.com/qgrimes, https://twitter....","[097 Sanchez Islands Apt. 393\nPort Tammy, AS ...","[The, digital, age, has, irrevocably, woven, i...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,The question of identity in the digital age is...,"[bosborne@gmail.com, crystalgarcia@hotmail.com]","[james71, bishoptanner, debra94]",[418 3MZ],"[2679537510, 608-399-3318x868]",[https://instagram.com/ronaldknight],"[PSC 1611, Box 6207\nAPO AA 90471, 4151 Michae...","[The, question, of, identity, in, the, digital...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,The flickering fluorescent lights of the unive...,[yfigueroa@yahoo.com],[scott92],[161-16-1975],[(769)972-8457x6377],[https://twitter.com/wallacedouglas],[],"[The, flickering, fluorescent, lights, of, the...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,## The Unexpected Detour: Finding Community in...,"[donnadennis@gmail.com, carterhannah@hotmail.com]",[michelelopez],[DPLW42035574485291],[789-542-6223],[https://facebook.com/karicarter],"[90611 Robert Plaza\nYangberg, OR 55838]","[The, Unexpected, Detour, Finding, Community, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
